<a href="https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/solasobambo-prog/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**The rule, in plain words:** a page is worth reviewing for a CTR fix if it has enough
impressions to trust its CTR (>= 500), and its CTR sits below the average CTR for its
own position tier. The bigger that gap, and the bigger the audience behind it, the
higher it ranks, since a small gap on a huge audience is a bigger recoverable
opportunity than the same gap on a handful of impressions.

**Reason code (one, per this rule):** `ctr_below_tier_visible`, meaning the page cleared
the volume floor and sits below its tier's expected CTR.

**Action label:** `review_ctr_snippet`, an editor should review the page's title, meta
description, or snippet for a possible CTR fix.

**Two signals checked below, one bucket table each, with n:**

1. **CTR vs position tier**, the signal behind FlyRank's real `needs_ctr_fix` flag
   (per the lane guide, section 4). Checked on `impressions_90d >= 100` (the trust floor
   established in ML-02).
2. **Impression volume**, the signal behind FlyRank's real `is_quick_win` flag. Checked
   as `ctr_gap` (actual CTR minus tier average) bucketed by `impressions_90d`, to see
   whether low volume produces noisier, less trustworthy gaps.

In [12]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

print("Login successful, token loaded from Colab Secrets.")

Login successful, token loaded from Colab Secrets.


In [13]:
from huggingface_hub import HfApi

api = HfApi()
files = api.list_repo_files("FlyRank/internship-warehouse", repo_type="dataset")
print(f"Found {len(files)} files in the warehouse repo.")
print(files[:10])

Found 24 files in the warehouse repo.
['.gitattributes', 'README.md', 'dim_clients.parquet', 'dim_content.parquet', 'fact_content_daily_performance/month=2025-01/data_0.parquet', 'fact_content_daily_performance/month=2025-02/data_0.parquet', 'fact_content_daily_performance/month=2025-03/data_0.parquet', 'fact_content_daily_performance/month=2025-04/data_0.parquet', 'fact_content_daily_performance/month=2025-05/data_0.parquet', 'fact_content_daily_performance/month=2025-06/data_0.parquet']


In [14]:
import os, subprocess

# Make sure we're in the repo root, no matter where this cell is run from.
# Safe to re-run: skips the clone if the repo is already there.
if not os.path.exists("data/raw/content_refresh_anonymized.csv"):
    if not os.path.isdir("flyrank-ml-internship"):
        subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/solasobambo-prog/flyrank-ml-internship.git"],
            check=True,
        )
    os.chdir("flyrank-ml-internship")

import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# ---- Signal 1: CTR vs position tier (behind FlyRank's needs_ctr_fix flag) ----
visible = df[df["impressions_90d"] >= 100].copy()
sig1 = visible.groupby("position_tier")["ctr"].agg(["mean", "count"]).sort_values("mean", ascending=False)
print("SIGNAL 1: CTR by position tier (impressions_90d >= 100), n per bucket")
print(sig1.round(3))
print(f"Spread: {sig1['mean'].max():.3f} down to {sig1['mean'].min():.3f} (~{sig1['mean'].max()/sig1['mean'].min():.1f}x)")
print("Verdict: CONFIRMED")
print()

# ---- Signal 2: impression volume (behind FlyRank's is_quick_win flag) ----
tier_avg_ctr = visible.groupby("position_tier")["ctr"].transform("mean")
visible["tier_avg_ctr"] = tier_avg_ctr
visible["ctr_gap"] = visible["ctr"] - visible["tier_avg_ctr"]

bins = [99, 500, 2000, float("inf")]
lbls = ["100-499", "500-1999", "2000+"]
visible["impression_bucket"] = pd.cut(visible["impressions_90d"], bins=bins, labels=lbls)

sig2 = visible.groupby("impression_bucket", observed=True)["ctr_gap"].agg(["count", "mean", "std", "min", "max"])
print("SIGNAL 2: ctr_gap by impression volume bucket (impressions_90d >= 100), n per bucket")
print(sig2.round(3))
print("Verdict: MIXED, low volume (100-499) is clearly noisier (higher std, extreme max),")
print("confirming a volume floor matters, but the pattern doesn't keep tightening past ~500.")

SIGNAL 1: CTR by position tier (impressions_90d >= 100), n per bucket
                mean  count
position_tier              
page_1         0.355   8633
top_3          0.334    533
striking       0.256   5903
page_3_5       0.142   6058
deep           0.055    879
Spread: 0.355 down to 0.055 (~6.4x)
Verdict: CONFIRMED

SIGNAL 2: ctr_gap by impression volume bucket (impressions_90d >= 100), n per bucket
                   count   mean    std    min     max
impression_bucket                                    
100-499             5291  0.011  0.583 -0.355  11.405
500-1999            6502 -0.039  0.288 -0.355   5.174
2000+              10213  0.019  0.313 -0.355   4.856
Verdict: MIXED, low volume (100-499) is clearly noisier (higher std, extreme max),
confirming a volume floor matters, but the pattern doesn't keep tightening past ~500.


## 2. Build the ranked queue (writes the CSV)

Tier average CTR is computed only on the trustworthy slice (`impressions_90d >= 100`,
Signal 1's floor), then mapped onto every row so pages with less history still get a
fair comparison point instead of a fabricated tier average of their own. The score
itself only fires (is non-zero) when both rule conditions hold: enough volume to trust
the number (`impressions_90d >= 500`, Signal 2's floor) and a CTR below that tier
average. Size of the score reflects both how far below tier average the page sits and
how much audience is behind that gap, no fitted weights, no future-window inputs.

In [15]:
import os

# tier average CTR is only trustworthy where impressions_90d >= 100 (Signal 1's floor)
trustworthy = df[df["impressions_90d"] >= 100]
tier_avg_ctr = trustworthy.groupby("position_tier")["ctr"].mean()

df["tier_avg_ctr"] = df["position_tier"].map(tier_avg_ctr)
df["ctr_gap"] = df["ctr"] - df["tier_avg_ctr"]

# the rule: visible enough to trust (Signal 2's floor) AND below tier average (Signal 1)
visible_flag = (df["impressions_90d"] >= 500).astype(int)
below_tier_flag = (df["ctr"] < df["tier_avg_ctr"]).astype(int)

df["score"] = visible_flag * below_tier_flag * df["impressions_90d"] * df["ctr_gap"].abs()

df["reason_code"] = ""
df["action_label"] = ""
flagged = df["score"] > 0
df.loc[flagged, "reason_code"] = "ctr_below_tier_visible"
df.loc[flagged, "action_label"] = "review_ctr_snippet"

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

out_cols = ["content_id", "position_tier", "impressions_90d", "ctr", "tier_avg_ctr",
            "ctr_gap", "score", "reason_code", "action_label"]

os.makedirs("work/outputs", exist_ok=True)
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Ranked queue written: {len(ranked):,} rows -> work/outputs/baseline_action_score.csv")
print(f"Flagged (score > 0): {flagged.sum():,} of {len(df):,} ({flagged.mean():.1%})")
print()
print("Top 10 preview:")
print(ranked[out_cols].head(10).to_string(index=False))


Ranked queue written: 30,000 rows -> work/outputs/baseline_action_score.csv
Flagged (score > 0): 11,016 of 30,000 (36.7%)

Top 10 preview:
          content_id position_tier  impressions_90d  ctr  tier_avg_ctr   ctr_gap         score            reason_code       action_label
content_5fe46e04994d        page_1           517715 0.14      0.354760 -0.214760 111184.288695 ctr_below_tier_visible review_ctr_snippet
content_8c19996aa890         top_3           509252 0.15      0.334128 -0.184128  93767.338236 ctr_below_tier_visible review_ctr_snippet
content_36ff89c8214e        page_1           295097 0.05      0.354760 -0.304760  89933.656438 ctr_below_tier_visible review_ctr_snippet
content_8451fc6f034d         top_3           272144 0.03      0.334128 -0.304128  82766.496060 ctr_below_tier_visible review_ctr_snippet
content_c8e9d6ab9013        page_1           208678 0.00      0.354760 -0.354760  74030.532830 ctr_below_tier_visible review_ctr_snippet
content_c84a0ab98e90        page_1     

## 3. Top-10 review

Pulled `content_type`, `word_count`, `content_age_days`, `days_since_last_update`, and
`trend_direction` for the top 10 to review with real context, not just the score.

1. **content_5fe46e04994d** (page_1, 517,715 impr): `review_ctr_snippet`, CTR 0.14 vs
   tier average 0.355, stale for 104 days and already trending down. Wrong if the
   snippet is fine and the real driver is a `NaN` word_count, meaning thin or
   unmeasured content, not a title/meta problem.
2. **content_8c19996aa890** (top_3, 509,252 impr): `review_ctr_snippet`, CTR 0.15 vs
   0.334, only 20 days since last update yet already trending down. Wrong if a recent
   change caused this dip and it is still settling, reviewing again in two weeks would
   tell.
3. **content_36ff89c8214e** (page_1, 295,097 impr): `review_ctr_snippet`, CTR 0.05 vs
   0.355, one of the largest gaps in the queue, and trend is merely "stable" so this
   has likely been this way a while. Wrong if this page's real intent (e.g. a
   definition query) never earns page_1-typical clicks regardless of snippet quality.
4. **content_8451fc6f034d** (top_3, 272,144 impr): `review_ctr_snippet`, CTR 0.03 vs
   0.334, the deepest gap here, but trend is "up". Wrong if it means recovery is
   already underway and review effort is better spent elsewhere first.
5. **content_c8e9d6ab9013** (page_1, 208,678 impr): `review_ctr_snippet`, CTR is
   exactly 0.00 against 208,678 impressions. Wrong if this is a tracking or
   attribution gap rather than a real snippet problem, zero clicks on that much
   visibility is unusual enough to double check the raw numbers before assuming it is
   content-fixable.
6. **content_c84a0ab98e90** (page_1, 223,271 impr): `review_ctr_snippet`, CTR 0.03 vs
   0.355, updated only 20 days ago yet trend is "stable", so a fix has not helped yet.
   Wrong if 20 days is too short a window for a snippet or title change to show up in
   search behavior.
7. **content_cb112fce36be** (page_1, 309,910 impr): `review_ctr_snippet`, CTR 0.16 vs
   0.355, stale 104 days and trending down, the shape my rule was built to catch.
   Wrong if page_1's average CTR is itself inflated by a handful of branded/high-intent
   queries that this particular page does not compete for.
8. **content_73c54f78c06a** (page_1, 213,963 impr): `review_ctr_snippet`, CTR 0.10 vs
   0.355, recently touched (20 days) but trend still "stable", not yet responding.
   Wrong if the update was cosmetic and did not touch title/meta at all.
9. **content_aaef01a50def** (page_1, 517,109 impr): `review_ctr_snippet`, CTR 0.25 is
   the closest to tier average of the top 10, flagged mainly because its 517k
   impressions inflate the score even though the gap itself (-0.10) is comparatively
   small. Wrong if this is really a low-priority page that only ranks this high
   because of volume, not because the opportunity is unusually large.
10. **content_1a9e894be2e2** (page_1, 416,180 impr): `review_ctr_snippet`, CTR 0.23 vs
    0.355, large audience and a "down" trend. Wrong if the decline is seasonal for
    this topic rather than a fixable snippet issue.

**Pattern worth flagging going into Section 4:** every single row in the top 10 is
`content_type == "keyword article"`, and several have a missing `word_count`. That is
not diversity, it is a concentration risk worth checking before trusting the queue.

In [16]:
# Pull extra context for the top 10 to ground the review above, not just the score.
top10 = ranked.head(10)
context_cols = ["content_id", "position_tier", "impressions_90d", "ctr", "tier_avg_ctr",
                "ctr_gap", "score", "content_type", "word_count", "content_age_days",
                "days_since_last_update", "trend_direction"]
print(top10[context_cols].to_string(index=False))

          content_id position_tier  impressions_90d  ctr  tier_avg_ctr   ctr_gap         score    content_type  word_count  content_age_days  days_since_last_update trend_direction
content_5fe46e04994d        page_1           517715 0.14      0.354760 -0.214760 111184.288695 keyword article         NaN               537                     104            down
content_8c19996aa890         top_3           509252 0.15      0.334128 -0.184128  93767.338236 keyword article      2895.0               445                      20            down
content_36ff89c8214e        page_1           295097 0.05      0.354760 -0.304760  89933.656438 keyword article         NaN               144                     104          stable
content_8451fc6f034d         top_3           272144 0.03      0.334128 -0.304128  82766.496060 keyword article      3528.0               280                      20              up
content_c8e9d6ab9013        page_1           208678 0.00      0.354760 -0.354760  74030.532830 

## 4. Weak picks + leakage check

**Weak picks, confirmed:**

- **content_c8e9d6ab9013** (#5, page_1, 208,678 impr, ctr exactly 0.00): checked how
  common this is, 2,538 rows across the dataset have `ctr == 0.00` with
  `impressions_90d >= 500`, so this is not a one-off. A page can get a large score here
  purely from a suspicious zero, not a genuine snippet problem. The rule cannot tell
  the difference between "real zero clicks" and "tracking gap," that is a real weak
  spot, not something I am inventing to fill this section.
- **content_aaef01a50def** (#9, page_1, 517,109 impr, ctr_gap only -0.10): the smallest
  gap in the top 10, ranked mainly because of volume, not because the opportunity is
  unusually large. The rule multiplies gap by impressions, so a huge audience can push
  a small, ordinary-looking gap above a genuinely large gap on fewer impressions. Worth
  a size-adjusted variant later (for example, gap alone above a volume floor, rather
  than gap times volume) to see if the ranking changes meaningfully.
- **content_type concentration:** `keyword article` is 90.7% of the full dataset but
  98.7% of everything the rule flags, and literally 100% of the top 50. Not as
  alarming as it first looked (this content type already dominates the dataset), but
  it does mean `comparison article` and `feedly article` are effectively locked out of
  the top of the queue even where they might have real opportunities, since neither
  content_type nor anything correlated with it appears in the score.

**Leakage and product-flag check:**

The score only touches `position_tier`, `impressions_90d`, `ctr`, and the
`tier_avg_ctr` computed from those same current-period columns, nothing future-window,
nothing label-derived. `trend_direction` and `trend_pct` exist in this dataset but were
never used inside the score, only mentioned in the Section 3 review as extra human
context. FlyRank's real product decision flags (`health_score`, `priority_score`,
`action_type`, `refresh_tier`, `needs_ctr_fix`, `is_quick_win`) are not present in this
starter dataset at all, confirmed by checking the columns directly, so there was
nothing to accidentally leak in even if I had reached for it.

In [17]:
# Weak-pick check 1: is content_type concentration in the flagged set worse than its
# overall share in the dataset, or just proportional to how common it already is?
overall_share = df["content_type"].value_counts(normalize=True)
flagged_share = df.loc[df["score"] > 0, "content_type"].value_counts(normalize=True)
top50_share = ranked.head(50)["content_type"].value_counts(normalize=True)

compare = pd.DataFrame({
    "overall_%": (overall_share * 100).round(1),
    "flagged_%": (flagged_share * 100).round(1),
    "top50_%": (top50_share * 100).round(1),
}).fillna(0.0)
print("Content type share: overall vs flagged (score > 0) vs top 50")
print(compare)
print()

# Weak-pick check 2: how common is the ctr == 0.00 pattern behind pick #5?
zero_ctr = df[(df["ctr"] == 0) & (df["impressions_90d"] >= 500)]
print(f"Rows with ctr == 0.00 and impressions_90d >= 500: {len(zero_ctr):,}")
print()

# Leakage / product-flag check: confirm the score never touches a label-derived,
# future-window, or product-decision column, and that those product columns
# are not even present in this dataset.
used_columns = ["position_tier", "impressions_90d", "ctr", "tier_avg_ctr"]
banned_columns = ["trend_direction", "trend_pct", "is_declining_label",
                   "health_score", "priority_score", "action_type", "refresh_tier"]

print("Columns the score actually uses:", used_columns)
present_banned = [c for c in banned_columns if c in df.columns]
print("Banned/product-flag columns present in dataset:", present_banned if present_banned else "none")
print("Banned columns used inside the score:", [c for c in used_columns if c in banned_columns] or "none")

Content type share: overall vs flagged (score > 0) vs top 50
                    overall_%  flagged_%  top50_%
content_type                                     
comparison article        2.3        0.6      0.0
feedly article            7.0        0.7      0.0
keyword article          90.7       98.7    100.0

Rows with ctr == 0.00 and impressions_90d >= 500: 2,538

Columns the score actually uses: ['position_tier', 'impressions_90d', 'ctr', 'tier_avg_ctr']
Banned/product-flag columns present in dataset: ['trend_direction', 'trend_pct']
Banned columns used inside the score: none


## Self-check

Self-check:

All four sections filled with real reasoning and real numbers: ✅
Runs top to bottom clean: ✅, I have confirmed every cell's output matches
No client names, URLs, or private queries: ✅, everything is pseudonymized content_ids and public dataset columns
Careful words throughout (observed, directional, decision-support), no causal claims about snippets fixing anything: ✅